# NCTB-SchoolText — corpus recon

Stage 1 of the pipeline. Downloads NCTB-SchoolText, profiles subject/grade coverage, and
quantifies OCR noise so the KG source text is chosen on evidence rather than assumption.

Source: https://data.mendeley.com/datasets/f3882ccczp/1 (CC BY 4.0)

**Kaggle setup:** this notebook downloads from Mendeley, so turn Internet ON in the
notebook settings sidebar (requires a phone-verified Kaggle account). If you can't enable
internet, download the zip locally and add it as a private Kaggle Dataset, then point
`ZIP_PATH` at `/kaggle/input/<your-dataset>/NCTB-SchoolText.zip`.

In [ ]:
import json, glob, zipfile, shutil, urllib.request
from pathlib import Path
import pandas as pd

MENDELEY_URL = "https://data.mendeley.com/public-files/datasets/f3882ccczp/files/54e66048-5d2b-47a0-a489-a0b104119476/file_downloaded"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
ZIP_PATH = WORK / "NCTB-SchoolText.zip"
CORPUS = WORK / "nctb"

# Mendeley's CDN 403s the default Python-urllib User-Agent. Any real UA gets through.
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0 Safari/537.36"

if not ZIP_PATH.exists():
    req = urllib.request.Request(MENDELEY_URL, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=120) as r, open(ZIP_PATH, "wb") as f:
        shutil.copyfileobj(r, f)

if not zipfile.is_zipfile(ZIP_PATH):
    ZIP_PATH.unlink()
    raise SystemExit(
        "Downloaded file is not a zip — the CDN likely served an error page.\n"
        "Fall back to: download the zip in a browser, add it as a private Kaggle Dataset,\n"
        "and set ZIP_PATH = Path('/kaggle/input/<your-dataset>/NCTB-SchoolText.zip')."
    )

if not CORPUS.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(CORPUS)

print(f"{ZIP_PATH.stat().st_size / 1e6:.1f} MB")
print(sorted(p.name for p in CORPUS.iterdir() if p.is_dir()))

In [ ]:
def load_class(class_dir):
    rows = []
    for subj_dir in sorted(glob.glob(f"{class_dir}/processed_chapters_*")):
        for f in sorted(glob.glob(f"{subj_dir}/*.jsonl")):
            with open(f, encoding="utf-8") as fh:
                for line in fh:
                    line = line.strip()
                    if line:
                        rows.append(json.loads(line))
    return pd.DataFrame(rows)

df = load_class(CORPUS / "classNineTen")
print(df.shape)
df.head(3)

## Coverage — which subjects are actually worth building on

In [ ]:
df["n_chars"] = df["text"].str.len()

coverage = (
    df.groupby("subject")
      .agg(chapters=("chapter_no", "nunique"),
           chunks=("chunk_id", "count"),
           chars=("n_chars", "sum"),
           median_chunk=("n_chars", "median"))
      .sort_values("chunks", ascending=False)
)
coverage.head(15)

## OCR noise

The corpus is OCR-extracted from textbook PDFs and is **not** clean, despite being
chunked and chapter-mapped. Two failure modes matter for KG construction:

1. **Page furniture** — headers/footers/press marks captured as their own chunks.
2. **Formula destruction** — equations, numerals and inline English get mangled into
   digit soup. This hits equation-heavy subjects hardest.

Bengali-script ratio is a cheap proxy for the second: real Bangla prose sits near 1.0, a
mangled formula block drops well below 0.5.

**Caveat:** the ratio test only means anything for Bangla-medium books. Applied blindly it
flags 100% of the English and English-grammar books and ~60% of Arabic as "mangled", which
is wrong — those are simply in another script. They're excluded below.

**Second caveat:** these are heuristics, not ground truth. They estimate the scale of the
problem; they don't certify any individual chunk. Eyeball the samples before trusting them.

In [ ]:
def bengali_ratio(s):
    letters = [c for c in s if c.isalnum()]
    if not letters:
        return 0.0
    return sum(1 for c in letters if "ঀ" <= c <= "৿") / len(letters)

# Books legitimately not in Bengali script — the ratio test is meaningless for these and
# flags 100% of the English books as mangled if applied blindly.
NON_BANGLA_SCRIPT = {"English", "English_grammar", "Arabic", "Sanskrit", "Pali"}

df["ben_ratio"] = df["text"].map(bengali_ratio)
df["is_furniture"] = df["n_chars"] < 80
df["is_mangled"] = (df["ben_ratio"] < 0.5) & ~df["subject"].isin(NON_BANGLA_SCRIPT)
df["is_junk"] = df["is_furniture"] | df["is_mangled"]

noise = (
    df.groupby("subject")
      .agg(chunks=("chunk_id", "count"),
           furniture_pct=("is_furniture", lambda s: round(s.mean() * 100, 1)),
           mangled_pct=("is_mangled", lambda s: round(s.mean() * 100, 1)),
           junk_pct=("is_junk", lambda s: round(s.mean() * 100, 1)))
      .assign(usable=lambda d: (d.chunks * (1 - d.junk_pct / 100)).astype(int))
      .sort_values("junk_pct", ascending=False)
)
noise.head(15)

In [ ]:
# Eyeball the worst chunks before trusting any of this — heuristics are not ground truth.
SUBJECT = "physics_secondary"

worst = (df[(df.subject == SUBJECT) & (df.n_chars > 40)]
         .nsmallest(5, "ben_ratio"))
for r in worst.itertuples():
    print(f"[{r.chunk_id}] ratio={r.ben_ratio:.2f}")
    print("   ", r.text[:200].replace("\n", " | "), "\n")

## Chapter boundary sanity check

`chapters_config_*.json` carries the page ranges used during extraction. A chapter whose
span is wildly out of line with its neighbours usually means a missed boundary, which
would put content under the wrong chapter node in the KG.

In [ ]:
# Config filenames are lowercase; the `subject` field in the JSONL is not consistently so
# ("Math" vs "physics_secondary"). Normalise when joining the two.
cfg_path = CORPUS / "classNineTen" / f"chapters_config_{SUBJECT.lower()}.json"
cfg = json.load(open(cfg_path, encoding="utf-8"))

spans = pd.DataFrame(cfg)
spans["span"] = spans.end_page - spans.start_page
spans["suspect"] = spans.span > 2.5 * spans.span.median()
spans

## Persist for the next stage

In [ ]:
clean = df[~df.is_junk].drop(columns=["is_furniture", "is_mangled", "is_junk"])
clean.to_parquet(WORK / "nctb_class9_10_clean.parquet", index=False)
coverage.to_csv(WORK / "coverage.csv")
noise.to_csv(WORK / "noise.csv")

print(f"kept {len(clean):,} of {len(df):,} chunks ({len(clean)/len(df)*100:.1f}%)")